# 01 - Looking at the raw data

This is the first notebook of the project. Before cleaning anything, the
goal here is just to **look** at the data and understand what condition
it is in.

I use three pandas functions on every table:
- `head()` to see some rows
- `info()` to see the columns, their type, and how many cells are not empty
- `isna().sum()` to count missing values in each column


In [1]:
import pandas as pd

RAW = "../../datasets/raw"
DIM = "../../datasets/dim"


## A quick look before anything else

Before checking each table, I open the production one and see how it
looks. Real data always comes with a few problems -- that's normal.


In [2]:
df_production = pd.read_csv(f"{RAW}/fact_production_raw.csv", encoding="utf-8-sig")
print(df_production.head(10))


         Date                Process MachineId     ToolId WorkOrder  \
0  2025-12-04    Hot Foil Stamping      HF-002  RS-HF-001   WO-1703   
1  2026-05-19           Blow Molding  ISBM-007  M-SOP-020   WO-8533   
2  2025-08-12           Blow Molding  ISBM-003  M-SOP-006   WO-6123   
3  2025-07-30           Blow Molding  ISBM-002  M-SOP-008   WO-5594   
4  2025-12-01      Injection Molding    IM-002  M-INJ-002   WO-2703   
5  2026-01-02      injection molding    IM-005  M-INJ-008   WO-4269   
6  2025-09-18           Blow Molding  ISBM-006  M-SOP-018   WO-7691   
7  2025-09-08         Blow Molding    ISBM-003  M-SOP-009   WO-6158   
8  2026-01-28           Blow Molding  ISBM-004  M-SOP-004   WO-6861   
9  2026-04-10           Blow Molding  ISBM-004  M-SOP-029   WO-6954   

             ProductId  ProductBatch StartTime   EndTime  PlannedQty  \
0      FR-008-LDPE-350   25338007040  13:39:00  19:40:00        3417   
1      FR-020-rPET-750   26139005340  21:28:00  12:53:00        6341   
2 

In [3]:
print(df_production["Process"].unique())


<ArrowStringArray>
['  Hot Foil Stamping  ',          'Blow Molding',     'Injection Molding',
     'injection molding',      '  Blow Molding  ',   '  Screen Printing  ',
 '  Injection Molding  ',     'Hot Foil Stamping',          'blow molding',
          'BLOW MOLDING',       'Screen Printing',     'INJECTION MOLDING',
     'HOT FOIL STAMPING',     'hot foil stamping',       'screen printing',
       'SCREEN PRINTING']
Length: 16, dtype: str


Notice: `Injection Molding` and `injection molding` show up as if they
were different processes, but they're the same thing written
differently. This is a classic real-data problem -- I fix it in the
next notebook (`02_data_cleaning_production.ipynb`) with
`.str.strip().str.title()`.


## 1. Production tables

Now I repeat the three commands (`head`, `info`, `isna().sum()`) for
each of the four production tables. Instead of copying the code four
times, I use a list of file names and a `for` loop.


In [4]:
production_files = [
    "fact_production_plan_raw.csv",
    "fact_production_raw.csv",
    "fact_downtime_raw.csv",
    "fact_material_consumption_raw.csv",
]

for file_name in production_files:
    path = f"{RAW}/{file_name}"
    df = pd.read_csv(path, encoding="utf-8-sig")

    print("=" * 70)
    print("Table:", file_name)
    print("Rows and columns:", df.shape)

    print("\nFirst rows:")
    print(df.head(5))

    print("\nInfo (columns, types, non-null):")
    print(df.info())

    print("\nMissing values per column:")
    print(df.isna().sum())


Table: fact_production_plan_raw.csv
Rows and columns: (9128, 10)

First rows:
         Date                Process MachineId     ToolId WorkOrder  \
0  2025-09-09         Blow Molding    ISBM-002  M-SOP-008   WO-5651   
1  2025-08-12      Injection Molding    IM-005  M-INJ-008   WO-4074   
2  2025-10-08           BLOW MOLDING  ISBM-003  M-SOP-025   WO-6199   
3  2026-01-05           Blow Molding  ISBM-005  M-SOP-010   WO-7328   
4  2026-04-09    Injection Molding      IM-003  M-INJ-003   WO-3382   

   PlannedQty StartTime   EndTime  PlannedHours            ProductId  
0        8830  18:23:00  09:25:00         15.03  FR-008-HDPE-PCR-350  
1       16335  22:46:00  20:27:00         21.68    TF-008-HDPE-28410  
2        4841  13:14:00  06:26:00         17.19      FR-025-PET-1000  
3       10067  21:51:00  15:15:00         17.41       FR-010-PET-400  
4       19922  10:31:00  00:05:00         13.57      TR-003-PP-24415  

Info (columns, types, non-null):
<class 'pandas.DataFrame'>
RangeInd

## 2. Quality control tables

These are the 8 tables from the quality lab: measurements, inspections,
and lot approval/rejection decisions for caps, bottles, and screen
printing ink.


In [5]:
quality_files = [
    "fact_cap_inspection_variable_cq_raw.csv",
    "fact_cap_attribute_inspection_cq_raw.csv",
    "fact_cap_disposition_lot_cq_raw.csv",
    "fact_bottle_inspection_variables_cq_raw.csv",
    "fact_bottle_attribute_inspection_cq_raw.csv",
    "fact_bottle_disposition_lot_cq_raw.csv",
    "fact_ink_attribute_inspection_cq_raw.csv",
    "fact_ink_disposition_lot_cq_raw.csv",
]

for file_name in quality_files:
    path = f"{RAW}/{file_name}"
    df = pd.read_csv(path, encoding="utf-8-sig")

    total_missing = df.isna().sum().sum()

    print("=" * 70)
    print("Table:", file_name)
    print("Rows and columns:", df.shape)
    print("Total missing values:", total_missing)
    print(df.head(5))


Table: fact_cap_inspection_variable_cq_raw.csv
Rows and columns: (83947, 33)
Total missing values: 8270
          ProductBatch WorkOrder ProductionDate    Shift MachineId     MoldId  \
0  LOTE-M-INJ-001-2490   WO-2248     2026-01-01  Shift 1    IM-001  M-INJ-001   
1  LOTE-M-INJ-001-2490   WO-2248     2026-01-01  SHIFT 1    IM-001  M-INJ-001   
2  LOTE-M-INJ-001-2490   WO-2248     2026-01-01  Shift 1    IM-001  M-INJ-001   
3  LOTE-M-INJ-001-2490   WO-2248     2026-01-01  SHIFT 1    IM-001  M-INJ-001   
4  LOTE-M-INJ-001-2490   WO-2248     2026-01-01  shift 1    IM-001  M-INJ-001   

               CapId Material    CapType Characteristic  ...      M6      M7  \
0  TR-001-PETG-24410     PETG  Screw Cap         Weight  ...  12.071  12.519   
1  TR-001-PETG-24410     PETG  Screw Cap         Weight  ...  11.890  11.385   
2  TR-001-PETG-24410     PETG  Screw Cap         Weight  ...  12.033  12.090   
3  TR-001-PETG-24410     PETG  Screw Cap         Weight  ...  12.011  12.089   
4  TR-001

## 3. Dimension tables (reference data)

These tables come from engineering (machine capacities, cap/bottle
specs, control plans), not from the shop floor -- so I expect them to
be cleaner than the production/quality tables. Let's check.


In [6]:
dimension_files = [
    "dim_masterbatch.csv",
    "dim_machine_setup.csv",
    "dim_cap.csv",
    "dim_cap_control_plan_cq.csv",
    "dim_bottle_control_plan_cq.csv",
    "dim_ink_control_plan_cq.csv",
    "dim_raw_material_control_plan.csv",
    "dim_customer.csv",
    "dim_supplier.csv",
]

for file_name in dimension_files:
    path = f"{DIM}/{file_name}"
    df = pd.read_csv(path, encoding="utf-8-sig")

    print("=" * 70)
    print("Table:", file_name)
    print("Rows and columns:", df.shape)
    print("Total missing values:", df.isna().sum().sum())


Table: dim_masterbatch.csv
Rows and columns: (132, 9)
Total missing values: 0
Table: dim_machine_setup.csv
Rows and columns: (38, 8)
Total missing values: 0
Table: dim_cap.csv
Rows and columns: (31, 15)
Total missing values: 0
Table: dim_cap_control_plan_cq.csv
Rows and columns: (10, 19)
Total missing values: 0
Table: dim_bottle_control_plan_cq.csv
Rows and columns: (13, 19)
Total missing values: 0
Table: dim_ink_control_plan_cq.csv
Rows and columns: (8, 19)
Total missing values: 0
Table: dim_raw_material_control_plan.csv
Rows and columns: (41, 12)
Total missing values: 0
Table: dim_customer.csv
Rows and columns: (14, 8)
Total missing values: 0
Table: dim_supplier.csv
Rows and columns: (8, 8)
Total missing values: 0


## 4. What I found

Looking at the results above, these are the problems I'll need to fix
in the next notebooks:

- **Categories written in different ways**, like `Injection Molding`
  and `injection molding` -- fixed with `.str.strip().str.title()`.
- **Missing values** in a few columns, mostly `OperatorId` and
  `MaintenanceTechnician`.
- **Negative numbers** where they shouldn't be, like `RejectedQty`
  (there's no such thing as rejecting a negative quantity of pieces).
- **Duplicate rows** in a few tables -- fixed with `drop_duplicates()`.
- Some "empty cells" actually have a space `" "` or a dash `"-"`
  typed in, instead of being truly empty. `isna()` doesn't catch these,
  so I need to handle that separately in the next notebook.
- The dimension tables really do arrive cleaner, as expected.

Cleaning and organizing data like this is usually the biggest chunk of
a data project -- exactly what the next notebook,
`02_data_cleaning_production.ipynb`, does.
